# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/test_newds` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [1]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv, find_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [17]:
load_dotenv(find_dotenv())

# API Config
os.environ["LITELLM_ILSP_EVAL_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["LITELLM_HOST"] = os.getenv("LITELLM_HOST")
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-26 13:58:42 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


In [18]:
# Ρυθμίσεις για να βρούμε τον κώδικα στο src
project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Προστέθηκε το {src_path} στο path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.")
except ImportError as e:
    logger.warning(f"⚠️ Δεν βρέθηκε η συνάρτηση/module. Έλεγξε τα ονόματα στο src. Error: {e}")

2026-01-26 13:58:44 - INFO - 🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.


## 2. Load and Prepare Dataset

In [52]:
#Script για εύκολη χρήση των νέων συναρτήσεων από το data_loader.py
from protipa_exams_dataset.data_loader import (
    load_protipa_dataset,  
    filter_dataset,
    apply_matching_processing
)

EVAL_MODE = 'open'

# Ρύθμιση φακέλου αποτελεσμάτων
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

logger.info(f"🚀 Ξεκινάει η διαδικασία σε Mode: {EVAL_MODE.upper()}...")

# --- Βήμα 1: Φόρτωση από Hugging Face ---
hf_dataset = load_protipa_dataset("PennyK98/test_newds", split="test") 
logger.info(f"✅ Loaded raw dataset from HF. Size: {len(hf_dataset)}")

# --- Βήμα 2: Φιλτράρισμα ---
filtered_list = filter_dataset(hf_dataset, mode=EVAL_MODE)
logger.info(f"✅ Filtered items ({EVAL_MODE}): {len(filtered_list)}")

# --- Βήμα 3: Μετατροπή σε DataFrame ---
df = pd.DataFrame(filtered_list)

# --- Βήμα 3b: ΔΙΟΡΘΩΣΗ ΤΩΝ ΕΙΚΟΝΩΝ ---
# Επειδή το HF μας δίνει PIL Objects (εικόνες) και εμείς θέλουμε Strings (ονόματα) για το JSON:
def extract_filename_from_object(val):
    # Αν είναι ήδη string (κείμενο), το κρατάμε
    if isinstance(val, str):
        return os.path.basename(val.replace('\\', '/'))
    # Αν είναι λίστα, το εφαρμόζουμε σε κάθε στοιχείο
    if isinstance(val, list):
        return [extract_filename_from_object(x) for x in val]
    # Αν είναι PIL Image (το PngImageFile που έβγαλε το error), παίρνουμε το filename του
    if hasattr(val, 'filename') and val.filename:
        return os.path.basename(val.filename.replace('\\', '/'))
    
    return None

logger.info("🖼️ Converting Image Objects to Filenames for JSON...")
if 'images' in df.columns:
    df['images'] = df['images'].apply(extract_filename_from_object)

# --- Βήμα 4: Διόρθωση Matching & Καθαρισμός ---
logger.info("🔄 Applying Matching Fix & Cleaning Paths...")
df = apply_matching_processing(df) 

# --- Βήμα 5: Αποθήκευση ---
if EVAL_MODE == 'closed':
    json_filename = "full_dataset_for_eval_closed.json" 
else:
    json_filename = "full_dataset_for_eval_open.json" 

full_data_path = results_dir / json_filename

if not df.empty:
    df.to_json(full_data_path, orient="records", force_ascii=False, indent=4)
    logger.info(f"💾 Saved {EVAL_MODE} dataset to: {json_filename}")
    logger.info(f"✅ Questions ready for evaluation: {len(df)}")
else:
    logger.warning("⚠️ Το DataFrame είναι άδειο! Δεν αποθηκεύτηκε τίποτα.")

2026-01-26 14:53:31 - INFO - 🚀 Ξεκινάει η διαδικασία σε Mode: OPEN...
2026-01-26 14:53:31 - INFO - Loading dataset from Hugging Face: PennyK98/test_newds
2026-01-26 14:53:34 - INFO - ✅ Loaded raw dataset from HF. Size: 1646



🔍 [LOCAL DEBUG] Filtering for mode: OPEN...


2026-01-26 14:53:34 - INFO - ✅ Filtered items (open): 278
2026-01-26 14:53:34 - INFO - 🖼️ Converting Image Objects to Filenames for JSON...
2026-01-26 14:53:34 - INFO - 🔄 Applying Matching Fix & Cleaning Paths...
2026-01-26 14:53:34 - INFO - 💾 Saved open dataset to: full_dataset_for_eval_open.json
2026-01-26 14:53:34 - INFO - ✅ Questions ready for evaluation: 278


------------------------------
📊 REPORT FOR OPEN:
   ✅ Kept: 278
   ❌ Wrong Subject: 0
   ❌ Wrong Format/Logic: 1368
------------------------------


## 3. Define Evaluation Task Template

In [53]:
full_path = (results_dir / json_filename).resolve()
path_str = str(full_path).replace('\\', '/')
logger.info(f"Setting up task with data file: {path_str}")

if EVAL_MODE == 'closed':
    task_config = {
        "task": "greek_protipa_exams_closed",
        "dataset_path": "json",
        "num_fewshot": 0,      
        "dataset_kwargs": {
                "data_files": path_str
            },
        "test_split": "train",
        "output_type": "generate_until",
        "doc_to_text": (
            "{% if input %}{{input}}\n{% endif %}"
            "Ερώτηση: {{question}}\n"
            "Επιλογές:\n"
            "{% for choice in choices %}"
            "{{loop.index0}}. {{choice}}\n"
            "{% endfor %}\n"
            "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
            "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής (π.χ. 0, 1, 2, 3...).\n"
            "ΠΡΟΣΟΧΗ: Ο αριθμός '2' που χρησιμοποιείται στα παραδείγματα παρακάτω είναι ΤΥΧΑΙΟΣ και αφορά μόνο τη ΜΟΡΦΗ της απάντησης εδώ.\n"
            "Η σωστή απάντηση εξαρτάται αποκλειστικά από την ερώτηση και μπορεί να είναι οποιοσδήποτε αριθμός.\n"
            "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
            "Παραδείγματα Μορφής:\n"
            "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
            "❌ ΛΑΘΟΣ: \"(2)\"\n"
            "✅ ΣΩΣΤΟ: 2 (ή 0 ή 1 ή 3... ανάλογα με τη σωστή επιλογή)\n\n"
            "Απάντηση: "
        ),
        "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
        "generation_kwargs": {
            "until": ["\n"],
            "max_gen_toks": 50,
            "do_sample": False,
            "temperature": 0.0 
        },
        "filter_list": [
            {
                "name": "strict-match",
                "filter": [
                    {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                    {"function": "take_first"}
                ]
            }
        ],
        "metric_list": [
            {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
        ]
    }

else: # EVAL_MODE == 'open'
    task_config = {
        "task": "greek_protipa_exams_open",
        "dataset_path": "json",
        "num_fewshot": 0,      
        "dataset_kwargs": {
            "data_files": path_str
        },
        "test_split": "train",
        "output_type": "generate_until",
        "doc_to_text": (
            "Δίνεται η παρακάτω ερώτηση από σχολικές εξετάσεις.\n"
            "Αν είναι ερώτηση ανάπτυξης, δώσε μια ολοκληρωμένη και τεκμηριωμένη απάντηση.\n"
            "Αν είναι ερώτηση συμπλήρωσης κενών, γράψε τη σωστή λέξη ή τη σωστή φράση που λείπει.\n\n"
            "{% if input %}Πλαίσιο/Κείμενο: {{input}}\n{% endif %}"
            "Ερώτηση: {{question}}\n\n"
            "Απάντηση:"
        ),
        "doc_to_target": "{{ answer_text }}",
        "generation_kwargs": {
            "until": ["Ερώτηση:", "---"], # Σταματάει αν πάει να ξεκινήσει νέα ερώτηση
            "max_gen_toks": 512,          # Δίνουμε χώρο για ανάπτυξη
            "do_sample": False,
            "temperature": 0.0 
        },
        "metric_list": [
            {
                "metric": "bleu", 
                "aggregation": "mean", 
                "higher_is_better": True
            },
            {
                "metric": "chrf", 
                "aggregation": "mean", 
                "higher_is_better": True
            }
        ]
    }

# --- Αποθήκευση και Φόρτωση ---
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)

yaml_filename = f"greek_protipa_{EVAL_MODE}.yaml"

with open(task_dir / yaml_filename, "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_name = f"greek_protipa_exams_{EVAL_MODE}"
task_dict = {task_name: custom_task}

logger.info(f"Evaluation task '{task_name}' defined successfully.")

2026-01-26 14:53:38 - INFO - Setting up task with data file: C:/Users/panag/Desktop/ΙΕΛ ΕΡΓΑΣΙΑ/Εξετάσεις_γλωσσομάθειας/Πρότυπα_Πειραματικά/results/full_dataset_for_eval_open.json


Generating train split: 0 examples [00:00, ? examples/s]

2026-01-26 14:53:38 - INFO - Evaluation task 'greek_protipa_exams_open' defined successfully.


## 4. Run Evaluation

In [54]:
from protipa_exams_dataset.evaluation import run_evaluation

comparison_results = {}
all_samples = {}

EVAL_LIMIT = 20  # Adjust this to run more/less samples

for model_name in models_to_test:
    
    results = run_evaluation(
        model_name=model_name,
        api_base=api_base,
        task_dict=task_dict,
        eval_limit=EVAL_LIMIT
    )

    if results is None:
        continue

    try:
        scores = results['results'][task_name]
        comparison_results[model_name] = scores
        
        if 'samples' in results and task_name in results['samples']:
            samples = results['samples'][task_name]
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    if EVAL_MODE == 'open':
                        ground_truth = str(doc.get('answer_text', 'N/A'))
                    else:
                        ground_truth = str(doc.get('answer_index', 'N/A'))
                    img_paths = doc.get('image_paths', [])
                    if isinstance(img_paths, list) and len(img_paths) > 0:
                        image_str = str(img_paths[0])
                    else:
                        image_str = ""
                    raw_choices = doc.get('choices', [])
                    if isinstance(raw_choices, list):
                        choices_str = " | ".join([str(c) for c in raw_choices])
                    else:
                        choices_str = str(raw_choices)
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Format": doc.get('format', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('admission_level', 'N/A'),
                        "Ground Truth": ground_truth,
                        "Choices": choices_str,
                        "Multimodality": doc.get('multimodality', 'no'),
                        "Image Path": image_str,
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        if EVAL_MODE == 'open':
            chrf = scores.get('chrf', scores.get('score', 0))
            bleu = scores.get('bleu', 0)
            
            logger.info(f"✅ Success! {model_name} Results:")
            logger.info(f"   🔹 ChrF: {chrf:.4f}")
            logger.info(f"   🔹 BLEU: {bleu:.4f}")
            
        else:
            acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
            logger.info(f"✅ Success! {model_name} Accuracy: {acc:.2%}")

        logger.info("⏳ Waiting 2 seconds before next model...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"❌ Error processing results for {model_name}: {e}")
        logger.error(traceback.format_exc())

2026-01-26 14:53:41 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-26 14:53:41 - INFO - Using max length 2048 - 1
2026-01-26 14:53:41 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-26 14:53:41 - INFO - Using tokenizer None
2026-01-26 14:53:41 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-26 14:53:41 - INFO - Building contexts for greek_protipa_exams_open on rank 0...

100%|██████████| 20/20 [00:00<00:00, 875.82it/s]
2026-01-26 14:53:41 - INFO - Running generate_until requests
2026-01-26 14:53:41 - INFO - Tokenized requests are disabled. Context + generation length is not checked.





















Requesting API: 100%|██████████| 20/20 [03:07<00:00,  9.36s/it]
2026-01-26 14:56:48 - INFO - ✅ Success! gemma3-27b-it Results:
2026-01-26 14:56:48 - INFO -    🔹 ChrF: 0.0000
2026-01-26 14:56:48 - INFO -    🔹 BLEU: 0

## 5. Results Table

In [55]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():

        row = {
            "ID": idx,
            "Subject": data.get("Subject", "N/A"),
            "Format": data.get("Format", "N/A"),
            "Multimodality": data.get("Multimodality", "no"),
            "Image Path": data.get("Image Path", ""),
            "Year": data.get("Year", "N/A"),
            "Level": data.get("Level", "N/A"),
            "Question": data.get("Question", ""),
            "Choices": data.get("Choices", ""),
            "Ground Truth": data.get("Ground Truth", "")
        }
        
        for m in models_to_test:
            model_col_name = m.split("/")[-1]
            
            # Τα predictions είναι μέσα σε nested λεξικό, οπότε το παίρνουμε σωστά
            predictions = data.get("Model Predictions", {})
            pred = predictions.get(m, "N/A")
            
            row[f"{model_col_name}_pred"] = pred
            
        table_data.append(row)

    df_results = pd.DataFrame(table_data)

    # Δημιουργία ονόματος αρχείου
    model_names_str = "_".join([m.split("/")[-1] for m in models_to_test])
    # Χρησιμοποιούμε το task_name για να είναι ξεκάθαρο (open/closed)
    filename = f"eval_results_{model_names_str}_{EVAL_MODE}.csv" 
    results_file = results_dir / filename

    # Αποθήκευση
    df_results.to_csv(results_file, index=False, encoding='utf-8-sig')
    logger.info(f"💾 Table saved successfully to: {results_file}")

    display(df_results.head())

else:
    logger.warning("⚠️ No samples collected to save!")

2026-01-26 14:58:43 - INFO - 💾 Table saved successfully to: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\eval_results_gemma3-27b-it_krikri-dpo-context_open.csv


,ID,Subject,Format,Multimodality,Image Path,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,greek_language,fill_in_the_gaps,no,,2019,lyceum,Στο παρακάτω απόσπασμα από το κείμενο Β να μετ...,,"[""Διείσδυσε"", ""αφουγκράστηκε"", ""καινοτόμησε"", ...","Διεισδύει **(διεισδύσε)** στην κοινωνία, αφουγ...","Στο απόσπασμα του κειμένου, τα ρήματα στον αόρ..."
1,1,mathematics,open_ended,no,,2019,lyceum,"Η διαφορά των εμβαδών δύο τετραγώνων, με πλευρ...",,$x^2 - y^2 = 13 \Leftrightarrow (x + y)(x - y)...,Αυτό είναι ένα πρόβλημα άλγεβρας που μπορεί να...,Η ερώτηση αυτή μπορεί να αντιμετωπιστεί με μαθ...
2,2,mathematics,open_ended,no,,2019,lyceum,Το ΑΒΓΔ είναι ορθογώνιο με ΑΔ = 1. Αν ΑΕ = 2 κ...,,Συμβολίζουμε $ΕΒ = x$. Αφού ΑΒΓΔ και ΕΒΓΖ είνα...,Απάντηση:\n\nΑφού το ορθογώνιο ΕΒΓΖ είναι όμοι...,Για να βρούμε την πλευρά $ΕΒ$ του ομοιόμορφου ...
3,3,mathematics,open_ended,no,,2019,lyceum,"Για κάθε ζεύγος πραγματικών αριθμών (α, β) ορί...",,$\alpha \beta = \alpha\beta \Leftrightarrow \a...,Η ερώτηση είναι ερώτηση ανάπτυξης και απαιτεί ...,Για να βρούμε πότε η νέα πράξη $*$ ταυτίζεται ...
4,4,mathematics,open_ended,no,,2019,lyceum,"Για κάθε ζεύγος πραγματικών αριθμών (α, β) ορί...",,$\begin{cases} x 10 = 75 \\ x y = 19 \end{case...,Ας αναλύσουμε τις δοθείσες σχέσεις χρησιμοποιώ...,"Για να λύσουμε το πρόβλημα, θα χρησιμοποιήσουμ..."


## 6. Performance Summary

In [56]:
if comparison_results:
    summary_data = []
    
    for model, metrics in comparison_results.items():
        row = {"Model": model}
        
        if EVAL_MODE == 'open':
            chrf = metrics.get('chrf', metrics.get('score', 0))
            bleu = metrics.get('bleu', 0)
            
            row['ChrF'] = chrf
            row['BLEU'] = bleu
        else:
            acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
            row['Accuracy'] = acc
        
        summary_data.append(row)
    
    df_summary = pd.DataFrame(summary_data)
    
    # Αποθήκευση με δυναμικό όνομα για να μην σβήνουμε τα προηγούμενα!
    #summary_filename = f"evaluation_summary_scores_{EVAL_MODE}.csv"
    #summary_file = results_dir / summary_filename
    
    #df_summary.to_csv(summary_file, index=False)
    #logger.info(f"📊 Summary saved to: {summary_file}")

    print(f"\n=== ΤΕΛΙΚΗ ΒΑΘΜΟΛΟΓΙΑ ({EVAL_MODE.upper()}) ===")
    
    # Formatting για ωραία εκτύπωση
    df_display = df_summary.copy()
    if 'Accuracy' in df_display.columns:
        df_display['Accuracy'] = df_display['Accuracy'].apply(lambda x: f"{x:.2%}")
    if 'ChrF' in df_display.columns:
        df_display['ChrF'] = df_display['ChrF'].apply(lambda x: f"{x:.4f}")
    if 'BLEU' in df_display.columns:
        df_display['BLEU'] = df_display['BLEU'].apply(lambda x: f"{x:.4f}")
        
    display(df_display)

else:
    logger.warning("⚠️ No comparison results found to summarize.")


=== ΤΕΛΙΚΗ ΒΑΘΜΟΛΟΓΙΑ (OPEN) ===


,Model,ChrF,BLEU
0,gemma3-27b-it,0.0000,0.0000
1,krikri-dpo-context,0.0000,0.0000


In [57]:
# Φορτώνουμε το CSV που μόλις φτιάξαμε
if results_file.exists():
    df = pd.read_csv(results_file)
    print(f"📊 Φορτώθηκαν {len(df)} ερωτήσεις για ανάλυση.\n")

    if EVAL_MODE == 'closed':
        print("🔒 Mode is CLOSED: Calculating binary Accuracy (Correct/Incorrect)...")
        
        def normalize_val(val):
            """Καθαρίζει την τιμή για να γίνει σωστή σύγκριση."""
            s = str(val).strip()
            # Αν κατά λάθος έγινε 1.0 (float string), το κάνουμε 1
            if s.endswith(".0"):
                s = s[:-2]
            return s

        for model in models_to_test:
            col_name = f"{model.split('/')[-1]}_pred" 
            
            if col_name in df.columns:
                gt_clean = df["Ground Truth"].apply(normalize_val)
                pred_clean = df[col_name].apply(normalize_val)
                
                df[f"{model.split('/')[-1]}_correct"] = (gt_clean == pred_clean).astype(int)
                print(f"   ✅ Calculated accuracy column for: {model}")

    else:
        print("🔓 Mode is OPEN: Calculating ChrF & BLEU metrics per row...")
        
        try:
            import evaluate
            # Φορτώνουμε τις μετρικές
            chrf = evaluate.load("chrf")
            bleu = evaluate.load("bleu")
            
            for model in models_to_test:
                model_short = model.split('/')[-1]
                col_name = f"{model_short}_pred"
                
                if col_name in df.columns:
                    chrf_scores = []
                    bleu_scores = []
                    
                    # Υπολογισμός για κάθε γραμμή ξεχωριστά
                    for index, row in df.iterrows():
                        pred = str(row[col_name])
                        ref = str(row["Ground Truth"])
                        
                        # ChrF Score
                        res_chrf = chrf.compute(predictions=[pred], references=[[ref]])
                        chrf_scores.append(res_chrf['score']) 
                        
                        # BLEU Score
                        res_bleu = bleu.compute(predictions=[pred], references=[[ref]])
                        bleu_scores.append(res_bleu['bleu'] * 100) 
                    
                    # Αποθήκευση στο DataFrame
                    df[f"{model_short}_chrf"] = chrf_scores
                    df[f"{model_short}_bleu"] = bleu_scores
                    print(f"   ✅ Calculated ChrF & BLEU columns for: {model}")
                    
        except ImportError:
            print("⚠️ Η βιβλιοθήκη 'evaluate' λείπει. Δεν υπολογίστηκαν scores ανά γραμμή.")
        except Exception as e:
            print(f"❌ Error during metrics calculation: {e}")

    # --- ΑΠΟΘΗΚΕΥΣΗ ---
    df.to_csv(results_file, index=False, encoding='utf-8-sig')
    print(f"\n💾 Updated CSV with scores saved to: {results_file}")

else:
    print("⚠️ Δεν βρέθηκε το αρχείο αποτελεσμάτων.")

📊 Φορτώθηκαν 20 ερωτήσεις για ανάλυση.

🔓 Mode is OPEN: Calculating ChrF & BLEU metrics per row...
   ✅ Calculated ChrF & BLEU columns for: gemma3-27b-it
   ✅ Calculated ChrF & BLEU columns for: krikri-dpo-context

💾 Updated CSV with scores saved to: c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\results\eval_results_gemma3-27b-it_krikri-dpo-context_open.csv


Ανάλυση RQ1: Ακρίβεια ανά μάθημα

In [58]:
# Ανάλυση RQ1: Ενοποιημένη Απόδοση ανά Μάθημα (Closed & Open)
print("\n" + "="*60)
print("🏆 RQ1: PERFORMANCE PER SUBJECT")
print("="*60)

# --- ΒΗΜΑ 1: ΣΥΓΧΩΝΕΥΣΗ (MERGE) ΑΡΧΕΙΩΝ ---
all_files = list(results_dir.glob("eval_results_*.csv"))

df_list = []
for filename in all_files:
    # Αποφεύγουμε τα summary files για να μην έχουμε διπλότυπα
    if "summary" in filename.name:
        continue
    try:
        temp_df = pd.read_csv(filename)
        df_list.append(temp_df)
        print(f"✅ Loaded: {filename.name}")
    except Exception as e:
        print(f"⚠️ Error loading {filename.name}: {e}")

if df_list:
    df = pd.concat(df_list, ignore_index=True)
    print(f"📊 Total merged rows: {len(df)}")
else:
    print("⚠️ Δεν βρέθηκαν αρχεία αποτελεσμάτων!")
    df = pd.DataFrame()

# --- ΒΗΜΑ 2: ΥΠΟΛΟΓΙΣΜΟΣ METRICS (Acc, ChrF, BLEU) ---

if not df.empty:
    # Εντοπισμός στηλών
    acc_cols = [c for c in df.columns if c.endswith('_correct')]
    chrf_cols = [c for c in df.columns if c.endswith('_chrf')]
    bleu_cols = [c for c in df.columns if c.endswith('_bleu')]

    group_col = "Subject" if "Subject" in df.columns else "subject"
    
    # A. Accuracy (Closed)
    if acc_cols:
        df_acc = df.groupby("Subject")[acc_cols].mean() * 100
        df_acc = df_acc.rename(columns={c: f"{c.replace('_correct', '')} (Acc %)" for c in acc_cols})
    else:
        df_acc = pd.DataFrame()

    # B. ChrF (Open)
    if chrf_cols:
        scale_factor = 100 if (df[chrf_cols].max().max() <= 1.0) else 1
        df_chrf = df.groupby("Subject")[chrf_cols].mean() * scale_factor
        df_chrf = df_chrf.rename(columns={c: f"{c.replace('_chrf', '')} (ChrF)" for c in chrf_cols})
    else:
        df_chrf = pd.DataFrame()

    # C. BLEU (Open) 
    if bleu_cols:
        scale_factor = 100 if (df[bleu_cols].max().max() <= 1.0) else 1
        df_bleu = df.groupby("Subject")[bleu_cols].mean() * scale_factor
        df_bleu = df_bleu.rename(columns={c: f"{c.replace('_bleu', '')} (BLEU)" for c in bleu_cols})
    else:
        df_bleu = pd.DataFrame()

    # D. Πλήθος Ερωτήσεων
    df_count = df.groupby("Subject").size().to_frame("Total Qs")

    # --- ΒΗΜΑ 3: ΤΕΛΙΚΟΣ ΠΙΝΑΚΑΣ ---
    rq1_final = df_count.join(df_acc, how='outer')\
                        .join(df_chrf, how='outer')\
                        .join(df_bleu, how='outer')

    # Formatting
    rq1_final = rq1_final.round(2).fillna('-')

    display(rq1_final)
    
    # Αποθήκευση του τελικού πίνακα για το Paper
    #rq1_csv_path = results_dir / "RQ1_final_table.csv"
    #rq1_final.to_csv(rq1_csv_path)
    #print(f"💾 Table saved to: {rq1_csv_path}")

else:
    logger.warning("⚠️ DataFrame is empty. Cannot process RQ1.")


🏆 RQ1: PERFORMANCE PER SUBJECT
✅ Loaded: eval_results_gemma3-27b-it_krikri-dpo-context_closed.csv
✅ Loaded: eval_results_gemma3-27b-it_krikri-dpo-context_open.csv
📊 Total merged rows: 40


,Total Qs,gemma3-27b-it (Acc %),krikri-dpo-context (Acc %),gemma3-27b-it (ChrF),krikri-dpo-context (ChrF),gemma3-27b-it (BLEU),krikri-dpo-context (BLEU)
Subject,,,,,,,
greek_language,32,75.0,45.0,39.15,7.60,0.00,0.00
mathematics,8,-,-,20.33,24.05,5.93,6.01


Ανάλυση RQ2: Απόδοση ανάλογα με το αν υπάρχει ή όχι εικόνα

In [59]:
# Ανάλυση RQ2: Multimodality (Εικόνα vs Κείμενο)
print("\n" + "=" * 60)
print("🏆 RQ2: IMPACT OF MULTIMODALITY")
print("=" * 60)

if 'df' in locals() and not df.empty:
    
    # 1. Εντοπισμός Στηλών
    acc_cols = [c for c in df.columns if c.endswith('_correct')]
    chrf_cols = [c for c in df.columns if c.endswith('_chrf')]
    bleu_cols = [c for c in df.columns if c.endswith('_bleu')]

    # 2. Υπολογισμός Metrics ανά 'Multimodality' (yes/no)
    
    # --- A. Accuracy (Closed Tasks) ---
    if acc_cols:
        df_acc = df.groupby("Multimodality")[acc_cols].mean() * 100
        df_acc = df_acc.rename(columns={c: f"{c.replace('_correct', '')} (Acc %)" for c in acc_cols})
    else:
        df_acc = pd.DataFrame()

    # --- B. ChrF (Open Tasks) ---
    if chrf_cols:
        scale_factor = 100 if (df[chrf_cols].max().max() <= 1.0) else 1
        df_chrf = df.groupby("Multimodality")[chrf_cols].mean() * scale_factor
        df_chrf = df_chrf.rename(columns={c: f"{c.replace('_chrf', '')} (ChrF)" for c in chrf_cols})
    else:
        df_chrf = pd.DataFrame()

    # --- C. BLEU (Open Tasks) ---
    if bleu_cols:
        scale_factor = 100 if (df[bleu_cols].max().max() <= 1.0) else 1
        df_bleu = df.groupby("Multimodality")[bleu_cols].mean() * scale_factor
        df_bleu = df_bleu.rename(columns={c: f"{c.replace('_bleu', '')} (BLEU)" for c in bleu_cols})
    else:
        df_bleu = pd.DataFrame()

    # --- D. Πλήθος Ερωτήσεων ---
    df_count = df.groupby("Multimodality").size().to_frame("Total Qs")

    # 3. Τελικός Πίνακας RQ2
    # Ενώνουμε όλα τα DataFrames
    rq2_final = df_count.join(df_acc, how='outer')\
                        .join(df_chrf, how='outer')\
                        .join(df_bleu, how='outer')

    # Μορφοποίηση (2 δεκαδικά, παύλα στα κενά)
    rq2_final = rq2_final.round(2).fillna('-')

    display(rq2_final)

    # Αποθήκευση
    #rq2_csv_path = results_dir / "RQ2_multimodality_impact_full.csv"
    #rq2_final.to_csv(rq2_csv_path)
    #print(f"💾 RQ2 Table saved to: {rq2_csv_path}")

else:
    print("⚠️ Το DataFrame (df) είναι κενό. Τρέξτε πρώτα το κελί του RQ1 (Merge).")


🏆 RQ2: IMPACT OF MULTIMODALITY


,Total Qs,gemma3-27b-it (Acc %),krikri-dpo-context (Acc %),gemma3-27b-it (ChrF),krikri-dpo-context (ChrF),gemma3-27b-it (BLEU),krikri-dpo-context (BLEU)
Multimodality,,,,,,,
no,40,75.0,45.0,31.62,14.18,2.37,2.4


Ανάλυση RQ3: Ακρίβεια ανά είδος άσκησης 

In [60]:
# Ανάλυση RQ3: Ακρίβεια ανά Είδος Άσκησης
print("\n" + "=" * 60)
print("🏆 RQ3: PERFORMANCE PER FORMAT")
print("=" * 60)

if 'df' in locals() and not df.empty:
    
    # 1. Εντοπισμός Στηλών
    acc_cols = [c for c in df.columns if c.endswith('_correct')]
    chrf_cols = [c for c in df.columns if c.endswith('_chrf')]
    bleu_cols = [c for c in df.columns if c.endswith('_bleu')]

    # 2. Υπολογισμός Metrics ανά Format
    group_col = "Format"
    
    # --- A. Accuracy ---
    if acc_cols:
        df_acc = df.groupby(group_col)[acc_cols].mean() * 100
        df_acc = df_acc.rename(columns={c: f"{c.replace('_correct', '')} (Acc %)" for c in acc_cols})
    else:
        df_acc = pd.DataFrame()

    # --- B. ChrF ---
    if chrf_cols:
        scale_factor = 100 if (df[chrf_cols].max().max() <= 1.0) else 1
        df_chrf = df.groupby(group_col)[chrf_cols].mean() * scale_factor
        df_chrf = df_chrf.rename(columns={c: f"{c.replace('_chrf', '')} (ChrF)" for c in chrf_cols})
    else:
        df_chrf = pd.DataFrame()

    # --- C. BLEU ---
    if bleu_cols:
        scale_factor = 100 if (df[bleu_cols].max().max() <= 1.0) else 1
        df_bleu = df.groupby(group_col)[bleu_cols].mean() * scale_factor
        df_bleu = df_bleu.rename(columns={c: f"{c.replace('_bleu', '')} (BLEU)" for c in bleu_cols})
    else:
        df_bleu = pd.DataFrame()

    # --- D. Πλήθος Ερωτήσεων ανά Τύπο ---
    df_count = df.groupby(group_col).size().to_frame("Total Qs")

    # 3. Τελικός Πίνακας RQ3
    # Ενώνουμε όλα τα DataFrames
    rq3_final = df_count.join(df_acc, how='outer')\
                        .join(df_chrf, how='outer')\
                        .join(df_bleu, how='outer')

    # Μορφοποίηση (2 δεκαδικά, παύλα στα κενά)
    rq3_final = rq3_final.round(2).fillna('-')

    display(rq3_final)

    # Αποθήκευση
    #rq3_csv_path = results_dir / "RQ3_performance_per_format_full.csv"
    #rq3_final.to_csv(rq3_csv_path)
    #print(f"💾 RQ3 Table saved to: {rq3_csv_path}")

else:
    print("⚠️ Το DataFrame (df) είναι κενό. Τρέξτε πρώτα το κελί του RQ1 (Merge).")


🏆 RQ3: PERFORMANCE PER FORMAT


,Total Qs,gemma3-27b-it (Acc %),krikri-dpo-context (Acc %),gemma3-27b-it (ChrF),krikri-dpo-context (ChrF),gemma3-27b-it (BLEU),krikri-dpo-context (BLEU)
Format,,,,,,,
fill_in_the_gaps,12,-,-,39.15,7.6,0.0,0.0
multiple_choice,20,75.0,45.0,-,-,-,-
open_ended,8,-,-,20.33,24.05,5.93,6.01


Διαθέσιμα μοντέλα

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
api_base = os.getenv("OPENAI_BASE_URL")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            # Αν το ID περιέχει κάποια από τις λέξεις κλειδιά
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

**ΠΑΡΑΤΗΡΗΣΕΙΣ**

○ Τα υπόλοιπα μοντέλα (πχ llama-3.1-8b, mistral-7b-instruct-v0.2) σκάνε με Bedrock error όταν τα τρέχω

***ΟΛΑ ΤΑ ΜΑΘΗΜΑΤΑ, all closed-type questions*** (RQ1: Comparative Performance Across Subjects)

○ **Στο σύνολο των δεδομένων**, το gemma έχει accuracy 61% ενώ το krikri είχε 39%.

○ Στα επιμέρους μαθήματα παρατηρούνται τα εξής:

| Subject | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| ΓΛΩΣΣΑ | 723 | 71.6% | 45.2% |
| ΘΡΗΣΚΕΥΤΙΚΑ | 90 | 77.8% | 64.4% |
| ΜΑΘΗΜΑΤΙΚΑ | 555 | 44.3% | 26.3% |

***Multimodality VS Κείμενο*** (RQ2)

○ **Στο σύνολο των δεδομένων** παρατηρούνται τα εξής:

| Multimodality | Total Questions | gemma | krikri |
| :--- | :---: | :---: | :---: |
| no | 1176 | 63.5% | 40.6% |
| yes | 192 | 45.3% | 28.1% |

○ Για το gemma έχουμε πτώση 18.2% με την παρουσία εικόνας, ενώ για το krikri έχουμε πτώση μόλις 12.5%

***Αll types of closed questions (multiple-choice, fill-in-the-gaps, true/false, matching)*** (RQ3: Performance Across Exercise Types)

○ Ο τωρινός κώδικας (task_config) είναι φτιαγμένος να ψάχνει για ένα index, δηλαδή έναν αριθμό (π.χ. 1, 2, 3). Αυτό δουλεύει στο Multiple Choice, στο True/False και στο Fill-in-the-gaps γιατί εκεί η σωστή απάντηση είναι "Επιλογή 1" ή "Επιλογή 2". Στο Matching, η απάντηση δεν είναι ένας αριθμός. Είναι μια λίστα με ζεύγη (["1-γ", "2-α", "3-ε", "4-β", "5-δ"]). Στο dataset, το πεδίο answer_index είναι κενό (None) γιατί δεν υπάρχει "μία σωστή επιλογή", αλλά ένας συνδυασμός. Καλό θα ήταν να εξαιρέσουμε το Matching από αυτό το πείραμα (Closed Types), καθώς χρειάζεται άλλο κώδικα αξιολόγησης και πρόκειται για ελάχιστα παραδείγματα.

○ **Στο σύνολο των δεδομένων**, στα διαφορετικά είδη ερωτήσεων κλειστού τύπου παρατηρούνται τα εξής:

| Exercise Type    | Total Exercises | gemma | krikri |
| :--- | :---: | :---: | :---: |
| Fill-in-the-gaps | 9               | 88.9% | 77.8%  |
| Matching         | 4               | 0.0%  | 0.0%   |
| Multiple Choice  | 1268            | 59.3% | 37.1%  |
| True/False       | 87              | 85.1% | 60.9%  |

Πρόβλημα με matching ερωτήσεις

In [67]:
import pprint

print(f"🔍 Searching for 'matching' in the original dataset ({len(hf_dataset)} items)...")

found_matching = False

for item in hf_dataset:
    # Παίρνουμε το format και καθαρίζουμε τυχόν κενά
    fmt = str(item.get('format', '')).strip()
    
    if fmt == 'matching':
        print("\n✅ ΒΡΕΘΗΚΕ MATCHING ΕΡΩΤΗΣΗ!")
        print("=" * 60)
        
        # Τυπώνουμε τα βασικά πεδία για να δεις τη δομή
        print(f"📌 ID:       {item.get('id')}")
        print(f"📚 Subject:  {item.get('subject')}")
        print(f"❓ Question: {item.get('question')}")
        
        print("\n🔠 Choices (Λίστα):")
        pprint.pprint(item.get('choices'))
        
        print("\n🎯 Answer (Ground Truth):")
        print(item.get('answer')) # ή answer_text ή answer_index ανάλογα το dataset
        
        print("\n🖼️ Images:")
        print(item.get('image_paths'))
        
        print("=" * 60)
        
        found_matching = True
        break # Σταματάμε μόλις βρούμε την πρώτη

if not found_matching:
    print("❌ Δεν βρέθηκε καμία ερώτηση με format 'matching' στο hf_dataset.")

🔍 Searching for 'matching' in the original dataset (1646 items)...

✅ ΒΡΕΘΗΚΕ MATCHING ΕΡΩΤΗΣΗ!
📌 ID:       greek_language_lyc_2019_1_8
📚 Subject:  greek_language
❓ Question: Να αντιστοιχίσετε τα σημεία στίξης (Στήλη Α) με αυτό που δηλώνουν (Στήλη Β):

1. Τάμπλετ φορτωμένα με εφημερίδες στην αραβική γλώσσα για τους πρόσφυγες! (θαυμαστικό)
2. Το παράρτημα της βιβλιοθήκης στο σχολείο της (11ο Δημοτικό) (παρένθεση)
3. η μικρή «καταβρόχθιζε» λογοτεχνία (εισαγωγικά)
4. Πώς φαντάζεται μια κοινωνία χωρίς βιβλιοθήκη; (ερωτηματικό)
5. «Πολύ μικρός διάβαζα περισσότερο. Τώρα λόγω του σχολείου…λογοτεχνικό βιβλίο» (εισαγωγικά)
α. επεξήγηση
β. ερώτηση
γ. θαυμασμό
δ. λόγια προσώπου που αναφέρονται κατά λέξη-ευθύς λόγος
ε. μεταφορικό λόγο

🔠 Choices (Λίστα):
[]

🎯 Answer (Ground Truth):
None

🖼️ Images:
None
